# DNAR flow solver demo

Ноутбук можно запускать и из локального checkout, и независимо в Google Colab. Первая code-cell умеет найти уже смонтированный репозиторий или скачать его через `git clone`, после чего solver встраивается в контракт `VolumeDataset -> VolumeSolver.solve_checked() -> evaluate()`: DNAR-слой кодирует задачи/агентов/совместимые пары в дискретные состояния, делает несколько processor steps и передает ранжированный датасет существующему построителю маршрутов и repair.

Если запускаете в Colab и репозиторий приватный, заранее задайте корректный `REPO_URL`/`REPO_BRANCH` в первой ячейке или используйте Colab secret/token в URL.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab/local bootstrap. If the repo is already checked out, this cell reuses it.
# For Colab, replace REPO_URL with your fork/private URL if needed.
REPO_URL = os.environ.get('DNAR_FLOW_REPO_URL', 'https://github.com/your-org/dnar.git')
REPO_BRANCH = os.environ.get('DNAR_FLOW_REPO_BRANCH', 'flows/Optimization-of-flows')
COLAB_ROOT = Path('/content')
CLONE_DIR = COLAB_ROOT / 'dnar' if COLAB_ROOT.exists() else Path.cwd() / 'dnar_checkout'

def _has_flow_repo(path: Path) -> bool:
    return (path / 'Optimization-of-flows' / 'src' / 'flowopt' / 'volume_core').exists()

candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent, CLONE_DIR]
FLOW_ROOT = None
for candidate in candidates:
    if _has_flow_repo(candidate):
        FLOW_ROOT = candidate / 'Optimization-of-flows'
        break

if FLOW_ROOT is None:
    if 'your-org/dnar.git' in REPO_URL:
        raise ValueError(
            'Set REPO_URL to the real repository URL before running independently in Colab, '
            'or set environment variable DNAR_FLOW_REPO_URL.'
        )
    if not CLONE_DIR.exists():
        subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, str(CLONE_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(CLONE_DIR), 'fetch', 'origin', REPO_BRANCH, '--depth', '1'], check=True)
        subprocess.run(['git', '-C', str(CLONE_DIR), 'checkout', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(CLONE_DIR), 'pull', '--ff-only'], check=True)
    FLOW_ROOT = CLONE_DIR / 'Optimization-of-flows'

sys.path.insert(0, str(FLOW_ROOT / 'src' / 'flowopt'))
print('FLOW_ROOT =', FLOW_ROOT)
print('Python import path added =', FLOW_ROOT / 'src' / 'flowopt')

In [ ]:
import json
from volume_core import VolumeDataset, DnarFlowConfig, DnarFlowVolumeSolver

def build_volume_demo_dataset(source_path: Path, output_path: Path) -> Path:
    """Convert the in-repo mass-routing sandbox into VolumeDataset fields."""
    payload = json.loads(source_path.read_text(encoding='utf-8'))
    agent_depots = dict(payload.get('metadata', {}).get('agent_depots', []))
    default_depot = next(iter(payload.get('metadata', {}).get('depot_node_ids', []) or ['']))

    for node in payload.get('graph', {}).get('nodes', []):
        if str(node.get('kind', '')).startswith('object'):
            cap = float(node.get('object_day_capacity_tons') or 0.0)
            node['object_day_capacity_volume_m3'] = cap if cap > 0 else 999.0
        else:
            node['object_day_capacity_volume_m3'] = 0.0

    for task in payload.get('tasks', []):
        task['volume_raw_m3'] = float(task.get('mass_tons') or 0.0)
        task['container_type'] = 'A'
        task['required_container_types'] = ['A']
        task['is_compactable'] = False
        task['requires_compact_d'] = False

    for agent in payload.get('agents', []):
        vehicle_type = str(agent.get('vehicle_type', ''))
        agent['depot_node_id'] = agent_depots.get(agent.get('agent_id'), default_depot)
        agent['is_active_work_1st_shoulder'] = True
        agent['cap_container_A'] = True
        agent['cap_container_B'] = '2' in vehicle_type
        agent['cap_container_C'] = True
        agent['cap_container_D'] = bool(agent.get('is_compact', False))
        agent['max_raw_volume_m3'] = float(agent.get('capacity_tons') or 0.0)
        agent['max_hours'] = 10.0
        agent['max_daily_km'] = 260.0
        agent['avg_speed_kmph'] = 35.0
        agent['compaction_coeff'] = 1.0

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(payload, ensure_ascii=False), encoding='utf-8')
    return output_path

raw_dataset_path = FLOW_ROOT / 'storage' / 'syntetic_data_gap_vrp_solver' / 'data' / 'dataset_sandbox_type2.json'
dataset_path = build_volume_demo_dataset(raw_dataset_path, FLOW_ROOT / 'notebooks' / 'local' / 'dnar_volume_demo_dataset.json')
dataset = VolumeDataset.from_json(dataset_path)
active_agents = sum(1 for agent in dataset.agents if agent.is_active)
compatible_edges = sum(1 for task in dataset.tasks for agent in dataset.agents if dataset.agent_can_take_task(task, agent))
print(f'dataset={dataset_path}')
print(f'tasks={len(dataset.tasks)}, agents={len(dataset.agents)}, active_agents={active_agents}, compatible_task_agent_edges={compatible_edges}, nodes={len(dataset.nodes)}')

In [ ]:
solver = DnarFlowVolumeSolver(DnarFlowConfig(
    processor_steps=4,
    hidden_size=16,
    max_runtime_sec=30.0,
    verbose=True,
))
solution = solver.solve_checked(dataset)
evaluation = dataset.evaluate(solution)
evaluation.as_dict()

In [ ]:
print('Algorithm:', solution.algorithm)
print('Trips:', len(solution.trips))
print('Unassigned:', len(solution.unassigned_task_ids), solution.unassigned_task_ids[:10])
print('First DNAR logs:')
for line in solution.solver_logs[:8]:
    print(' ', line)